In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv('data/mental_disorders.tsv', sep='\t')
df

In [ ]:
df_mesh = df.dropna(subset=['MESH'])
df_mesh = df_mesh.assign(MESH=df_mesh['MESH'].str.split(';')).explode('MESH')
df_mesh = df_mesh.reset_index(drop=True)
df_mesh

In [ ]:
df_mesh[['MESH']].drop_duplicates().to_csv('data/mental_disorders_mesh_list.txt', sep='\t', index=False)

In [ ]:
df_mesh_works = pd.read_csv('data/mental_disorders_mesh_works.csv', sep=',')
df_mesh_works.drop_duplicates(inplace=True)
df_mesh_works

In [ ]:
df_mesh_works[['mesh', 'work_id']].groupby('mesh').count()

In [ ]:
df_mesh = df_mesh.merge(df_mesh_works, how='inner', left_on='MESH', right_on='mesh')
df_mesh = df_mesh[['Cause ID', 'work_id', 'publication_year']].drop_duplicates()
df_mesh

In [ ]:
df_works = df_mesh.copy()
df_works.drop_duplicates(inplace=True)
df_works.reset_index(drop=True, inplace=True)
df_works

In [ ]:
df_works = df_works.groupby(['Cause ID', 'publication_year']).size().unstack(fill_value=0)
df_works.columns = [f'works_{col}' for col in df_works.columns]
df_works['works'] = df_works.sum(axis=1)
df_works.reset_index(inplace=True)
df_works

In [ ]:
df = df.merge(df_works, how='inner', on='Cause ID')
df

In [ ]:
df.to_csv('data/mental_disorders_works.tsv', index=False, sep='\t')